# Example 4 — Variant effect prediction with Shorkie (logSED)\nScore a variant by the change in predicted gene-body coverage: `logSED = log2(Σ_alt + 1) − log2(Σ_ref + 1)` averaged over tracks. Positive ⇒ alt increases predicted expression. This is the core metric in the eQTL / MPRA benchmarks.\n\n**Prereqs:** `data/download.sh --models finetuned`. (CLI version: `minimal_example/run_shorkie_variant.py`.)

In [1]:
# Resolve the released model directory via shorkie.config (works on the training
# cluster). External users: run `data/download.sh --models all` and set
# MODEL_OVERRIDE to that local dir, or edit config/paths.yaml:work_root.
import os, json, numpy as np
from shorkie import config
MODEL_OVERRIDE = os.environ.get("SHORKIE_MODELS")  # e.g. ./data_local/models
import pandas as pd, pysam, tensorflow as tf
from shorkie.models.ensemble import load_ensemble, make_input, ensemble_predict, logSED, logSED_per_track
from baskerville import gene as bgene

model_dir   = MODEL_OVERRIDE or str(config.path("models.shorkie_finetuned"))
params_file  = "../minimal_example/params.json"   # released params
targets_file = "../minimal_example/sheet.txt"      # 5215-track sheet
fasta = pysam.Fastafile(str(config.path("genome.fasta")))
gtf   = str(config.path("genome.gtf"))

2026-06-24 17:25:09.039141: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-24 17:25:09.081483: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-24 17:25:09.081509: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-24 17:25:09.082715: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-24 17:25:09.089689: I tensorflow/core/platform/cpu_feature_guar

2026-06-24 17:25:09.840411: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### Load the ensemble + the gene of interest

In [2]:
target_index = pd.read_csv(targets_file, index_col=0, sep="\t").index
import io, contextlib
with contextlib.redirect_stdout(io.StringIO()):   # silence 8-fold restore logs
    models = load_ensemble(model_dir, params_file, target_index, num_folds=8)
print('loaded 8-fold Shorkie ensemble')
m0 = models[0]
txome = bgene.Transcriptome(gtf)

CHROM, POS, REF, ALT, GENE = "chrI", 124373, "T", "C", "YAL016C-B"   # 1-based POS
gobj = txome.genes[[k for k in txome.genes if GENE in k][0]]
print(f"gene {GENE}: {gobj.chrom}:{gobj.span()}")

loaded 8-fold Shorkie ensemble


gene YAL016C-B: I:(124306, 124492)


### Build ref/alt windows and score

In [3]:
gc  = gobj.midpoint(); off = m0.model_strides[0]*m0.target_crops[0]; olen = m0.model_strides[0]*m0.target_lengths[0]
lo = max(POS - 16384 + 1, gc - off - olen + 1); hi = min(POS - 1, gc - off)
start = int((lo+hi)//2) if lo <= hi else int(gc - 16384//2); end = start + 16384
gene_slice = gobj.output_slice(start + off, int(olen), m0.model_strides[0], False)

x_ref = make_input(fasta, CHROM, start, end, 16384)
ci = POS - start - 1
x_alt = np.copy(x_ref.numpy()); x_alt[ci, :4] = 0.0; x_alt[ci, {"A":0,"C":1,"G":2,"T":3}[ALT]] = 1.0
x_alt = tf.constant(x_alt)

y_ref = ensemble_predict(models, x_ref); y_alt = ensemble_predict(models, x_alt)
score = logSED(y_ref, y_alt, gene_slice)
print(f"{CHROM}:{POS} {REF}>{ALT}  gene {GENE}   logSED = {score:+.4f}")
print("logSED > 0 -> alt increases predicted expression; < 0 -> decreases")

chrI:124373 T>C  gene YAL016C-B   logSED = +0.0557
logSED > 0 -> alt increases predicted expression; < 0 -> decreases


### Which tracks change most?

In [4]:
per_track = logSED_per_track(y_ref, y_alt, gene_slice)   # (5215,)
order = np.argsort(np.abs(per_track))[::-1][:5]
for i in order:
    print(f"  {target_index[i]:>10}  logSED={per_track[i]:+.4f}")

        1807  logSED=+0.2267
        1813  logSED=+0.2224
        2021  logSED=+0.2197
        1708  logSED=+0.2081
        1990  logSED=+0.1975


For the full benchmarks see `reproduction/figure_07` (eQTL) and `reproduction/figure_06` (MPRA); the released score TSVs (`data/download.sh --eqtl --mpra`) let you reproduce those figures without re-scoring.